# Strain compatibility, the operator way: $\operatorname{inc}\varepsilon = \nabla\times(\nabla\times\varepsilon)^{\mathsf{T}}$

The strain incompatibility of a symmetric strain field $\varepsilon$ is
$\operatorname{inc}\varepsilon = \nabla\times(\nabla\times\varepsilon)^{\mathsf{T}}$, and
compatibility is $\operatorname{inc}\varepsilon = 0$.  We derive its closed form **as
performed** — expand $\nabla$ only, keep $\varepsilon$ abstract, reduce the cross,
reassemble the operators — then verify it componentwise (vibe 000078).

In [ ]:
import tender as t
import tender.basis as tb
import tender.derivation as td

ws = t.Workspace()
eps = ws.field(r"\varepsilon", 2, symmetric=True)
nabla = t.nabla(ctx=ws.ctx)

## 1. Coordinate-free, $\varepsilon$ abstract

`@` is the dot, `%` the cross, `*` the tensor product.  `inc ε` is built
chart-free — no $\varepsilon_{xy}$ components appear.

In [ ]:
inc = nabla % (nabla % eps).transpose()
inc

## 2. Expand $\nabla$ only — the free-index interior

`expand_nabla` lowers each $\nabla$ to the frame form $e_i\,\partial_i$, giving
$\sum_{i,j} e_i\times(e_j\times\partial_i\partial_j\varepsilon)^{\mathsf{T}}$ — still
abstract in $\varepsilon$, only second derivatives $\partial_i\partial_j\varepsilon$.

In [ ]:
x, y, z = ws.coords("x", "y", "z")
cart = ws.chart(ws.wcs(), [x, y, z], [x, y, z])
interior = cart.expand_nabla(inc)
interior

## 3. Derive the $a\times B\times c$ cross-removal identity (in-codebase)

Rather than assert it, we prove $a\times B\times c$ from first principles — expand in
the basis, apply $c\otimes a = a\times I\times c + (a\cdot c)I$, reduce the
$\varepsilon$-$\varepsilon$ contraction, reassemble — then compose with the
transpose-cross helper $a\times(c\times E)^{\mathsf{T}} = -a\times E\times c$ ($E$
symmetric) into `id_inc`.

In [ ]:
co = tb.Variance.Covariant
basis = tb.wcs(ws.ctx)
a = t.tensor("a", 1, ctx=ws.ctx)
c = t.tensor("c", 1, ctx=ws.ctx)
B = t.tensor("B", 2, ctx=ws.ctx)
E = t.field("E", 2, ctx=ws.ctx, symmetric=True)
I = t.identity(ws.ctx)

def derive(initial, steps):
    d = td.Derivation(initial)
    for s in steps:
        d.step(s)
    return d.current

axIxb = derive(a % I % c, (
    lambda x: tb.expand_in_basis(x, basis, co),
    lambda x: tb.simplify_basis_cross(x, basis),
    td.contract_eps_pair, td.contract_delta,
    lambda x: tb.reassemble(x, basis)))
id_alt = td.Identity("axIxb_alt",
    td.fold_equal_addends(axIxb + a @ c * I), a % I % c + a @ c * I)
axBxc = derive(a % B % c, (
    lambda x: tb.expand_in_basis(x, basis, co),
    td.apply_identity(id_alt),
    lambda x: tb.expand_in_basis(x, basis, co),
    lambda x: tb.simplify_basis_cross(x, basis),
    lambda x: tb.simplify_basis_dot(x, basis),
    td.contract_delta, td.contract_eps_pair, td.contract_delta,
    td.contract_eps_pair, td.contract_delta,
    lambda x: tb.reassemble(x, basis)))
id_axBxc = td.Identity("axBxc", a % B % c, axBxc)
id_inc = td.Identity("inc", a % (c % E).transpose(),
    td.canonicalize(-td.apply_identity(id_axBxc)(a % E % c)))
id_axBxc.rhs

## 4. Reduce the cross, then reassemble into $\nabla$ operators

Applying `id_inc` folds the nested cross into a cross-free sum (Phase 1);
`reassemble_nabla` reads each frame-indexed $\partial$'s role and folds it back
into $\nabla$ operators (Phase 2), giving the closed identity
$\operatorname{inc}\varepsilon = -\nabla\nabla\theta + \Delta\theta\,I
 - (\nabla\nabla\!\cdot\!\cdot\,\varepsilon)I - \Delta\varepsilon
 + 2(\nabla\nabla\!\cdot\varepsilon)^{\mathrm{s}}$, $\theta = \operatorname{tr}\varepsilon$.

In [ ]:
phase1 = td.canonicalize(td.apply_identity(id_inc)(interior))
assert "times" not in phase1.latex()  # Phase-1 is cross-free
reass = cart.reassemble_nabla(phase1)
reass

## 4b. Classical form under the compatibility condition

$\operatorname{inc}\varepsilon = 0$ forces its trace to vanish too. Taking the
trace of the closed identity gives $-\nabla\cdot(\nabla\cdot\varepsilon)
+ \Delta\operatorname{tr}\varepsilon$, so the trace condition is exactly the
scalar identity $\nabla\cdot(\nabla\cdot\varepsilon) = \Delta\operatorname{tr}\varepsilon$.
Rewriting with it cancels the $\nabla\cdot(\nabla\cdot\varepsilon)\,I$ term
against $\Delta\operatorname{tr}(\varepsilon)\,I$, collapsing the closed form to
the classical Saint-Venant compatibility equations
$-\Delta\varepsilon - \nabla\nabla(\operatorname{tr}\varepsilon)
+ \nabla(\nabla\cdot\varepsilon) + (\nabla(\nabla\cdot\varepsilon))^{\mathsf{T}} = 0$.

In [ ]:
theta = t.tr(eps)
id_trace = td.Identity("inc_trace", nabla @ (nabla @ eps), t.laplacian(theta))
classical = td.apply_identity(id_trace)(reass)
classical

## 5. Verify the closed identity, component by component

Both sides are coordinate-free tensors, so the identity holds in every frame —
tender confirms it in a Cartesian and a cylindrical frame.

In [ ]:
def is_zero(chart, e):
    return td.simplify_scalars(td.canonicalize(chart.expand(e))).latex() == "0"

def closed(chart, eps):
    th = t.tr(eps)
    gg = chart.components(chart.grad(chart.grad(th)))
    de = chart.components(chart.div(chart.grad(eps)))
    gd = chart.components(chart.grad(chart.div(eps)))
    lap = chart.laplacian(th); dd = chart.div(chart.div(eps))
    m = [[None]*3 for _ in range(3)]
    for i in range(3):
        for j in range(3):
            r = -gg[i][j] - de[i][j] + gd[i][j] + gd[j][i]
            if i == j:
                r = r + lap - dd
            m[i][j] = r
    return m

def verify(chart, eps):
    inc_m = chart.components(chart.rot(chart.rot(eps).transpose()))
    rhs = closed(chart, eps)
    return all(is_zero(chart, chart.expand(inc_m[i][j]) - chart.expand(rhs[i][j]))
               for i in range(3) for j in range(3))

ws2 = t.Workspace(); eps2 = ws2.field(r"\varepsilon", 2, symmetric=True)
r, th, zc = ws2.coords("r", r"\theta", "z", nonneg=("r",))
cyl = ws2.chart(ws2.wcs(), [r, th, zc], [r*t.cos(th), r*t.sin(th), zc])
print("Cartesian  :", "all 9 components equal" if verify(cart, eps) else "MISMATCH")
print("Cylindrical:", "all 9 components equal" if verify(cyl, eps2) else "MISMATCH")